In [1]:
import time
import nest
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import neo
import quantities as pq
import re
from elephant.statistics import isi, cv, mean_firing_rate
from elephant.conversion import BinnedSpikeTrain
from elephant.spike_train_correlation import corrcoef
from pathlib import Path

import subprocess

POWER_DEVICES = ["0000:2a:00.1"]  # only one U55C device

def measure_board_power(device):
    """Return power (W) for a single board using xrt-smi, or None on failure."""
    try:
        out = subprocess.check_output(
            ["xrt-smi", "examine", "-d", device, "-r", "electrical"],
            text=True
        )
    except Exception as e:
        print(f"Failed to read power for {device}: {e}")
        return None

    m = re.search(r"^\s*Power\s+:\s*([\d.]+)\s*Watts", out, re.MULTILINE)
    if not m:
        print(f"Could not parse power for {device}")
        return None
    return float(m.group(1))

def measure_total_power():
    """Sum power over all devices in POWER_DEVICES."""
    readings = [measure_board_power(dev) for dev in POWER_DEVICES]
    readings = [p for p in readings if p is not None]
    return sum(readings) if readings else None

## import model implementation
import network
## import (default) parameters (network, simulation, stimulus)
from network_params import default_net_dict as net_dict
from sim_params import default_sim_dict as sim_dict
from stimulus_params import default_stim_dict as stim_dict

# Import library NeuroRing for FPGA and pyxrt
import neuroring
import pyxrt
from utils_binding import *   # provides .index and .bitstreamFile

# Create network and connect neurons
net = network.Network(sim_dict, net_dict, stim_dict)
net.create()
net.connect()

print(net.pops)

param_dict = {
    'dt': 0.1,
    'tau_m': 10.0,
    'tau_syn': 0.5,
    'C_m': 250.0,
    'E_L': -65.0,
    't_ref_steps': 20,
    'V_th_abs': -50.0,
    'V_reset_abs': -65.0,
}
# 1 for recording, 0 for not recording spike
record_status = 0

host = neuroring.NeuroRingHost(net, 8192, 7000, 10, 2, param_dict, record_status, "/home/miahafiz/NeuroRing/_build_dir.hw.NUM_8192.CORE_5.FREQ_300/krnl_neuroring_hw.xclbin")



              -- N E S T --
  Copyright (C) 2004 The NEST Initiative

 Version: 3.9.0
 Built: Oct  2 2025 06:57:01

 This program is provided AS IS and comes with
 NO WARRANTY. See the file LICENSE for details.

 Problems or suggestions?
   Visit https://www.nest-simulator.org

 Type 'nest.help()' to find out more about NEST.

Data will be written to: /home/miahafiz/NeuroRing/host_py/data/
  Directory already existed. Old data will be overwritten.


RNG seed: 55
Total number of virtual processes: 4
Creating neuronal populations.

Mar 12 16:56:01 SimulationManager::set_status [Info]: 
    Temporal resolution changed from 0.1 to 0.1 ms.
Connecting neuronal populations recurrently.

Mar 12 16:57:00 NodeManager::prepare_nodes [Info]: 
    Preparing 77169 nodes for simulation.
[NodeCollection(metadata=None, model=iaf_psc_exp, size=20683, first=1, last=20683), NodeCollection(metadata=None, model=iaf_psc_exp, size=5834, first=20684, last=26517), NodeCollection(metadata=None, model=iaf_psc_ex

In [2]:
host.initialize_devices()
print("Initialized devices")


Initialized device 0 with XCLBIN UUID: 3db5ba43-3398-c60a-9d14-49a299296b90
{'simulation_time': 1, 'amount_of_cores': 10, 'neuron_start': 1, 'neuron_total': 8192, 'device': None, 'xclbin': None, 'uuid': None, 'kernel_name': None, 'kernel': None, 'neuron_per_cu': 8192, 'synapse_total_per_cu': 7000, 'param_dict': {'dt': 0.1, 'tau_m': 10.0, 'tau_syn': 0.5, 'C_m': 250.0, 'E_L': -65.0, 't_ref_steps': 20, 'V_th_abs': -50.0, 'V_reset_abs': -65.0}, 'record_status': 0, 'synapseListHandle': None, 'header_words': 25600000, 'header_bytes': 102400000, 'tail_words_capacity': 114688000, 'tail_bytes_capacity': 458752000, 'bo_size': 561152000, 'core_id': 0}
Initialized kernel NeuroRing:{NeuroRing_0} and SynapseRouter:{SynapseRouter_0} on device <pyxrt.device object at 0x7fae3290ae70>
Allocated BO of 561152000 bytes (header 102400000, tail 458752000)
{'simulation_time': 1, 'amount_of_cores': 10, 'neuron_start': 8193, 'neuron_total': 8192, 'device': None, 'xclbin': None, 'uuid': None, 'kernel_name': None

In [ ]:
host.kernels_per_fpga[0][0].upload_synapse_list(host.synapse_fpga[0])
host.kernels_per_fpga[0][1].upload_synapse_list(host.synapse_fpga[1])
host.kernels_per_fpga[0][2].upload_synapse_list(host.synapse_fpga[2])
host.kernels_per_fpga[0][3].upload_synapse_list(host.synapse_fpga[3])
host.kernels_per_fpga[0][4].upload_synapse_list(host.synapse_fpga[4])

host.kernels_per_fpga[1][0].upload_synapse_list(host.synapse_fpga[5])
host.kernels_per_fpga[1][1].upload_synapse_list(host.synapse_fpga[6])
host.kernels_per_fpga[1][2].upload_synapse_list(host.synapse_fpga[7])
host.kernels_per_fpga[1][3].upload_synapse_list(host.synapse_fpga[8])
host.kernels_per_fpga[1][4].upload_synapse_list(host.synapse_fpga[9])


In [12]:
timestep = 100000
start_time = time.time()
host.kernels_per_fpga[0][0].run_neuroring(timestep)
host.kernels_per_fpga[0][0].run_synapserouter(timestep)
host.kernels_per_fpga[0][1].run_neuroring(timestep)
host.kernels_per_fpga[0][1].run_synapserouter(timestep)
host.kernels_per_fpga[0][2].run_neuroring(timestep)
host.kernels_per_fpga[0][2].run_synapserouter(timestep)
host.kernels_per_fpga[0][3].run_neuroring(timestep)
host.kernels_per_fpga[0][3].run_synapserouter(timestep)
host.kernels_per_fpga[0][4].run_neuroring(timestep)
host.kernels_per_fpga[0][4].run_synapserouter(timestep)
host.kernels_per_fpga[1][0].run_neuroring(timestep)
host.kernels_per_fpga[1][0].run_synapserouter(timestep)
host.kernels_per_fpga[1][1].run_neuroring(timestep)
host.kernels_per_fpga[1][1].run_synapserouter(timestep)
host.kernels_per_fpga[1][2].run_neuroring(timestep)
host.kernels_per_fpga[1][2].run_synapserouter(timestep)
host.kernels_per_fpga[1][3].run_neuroring(timestep)
host.kernels_per_fpga[1][3].run_synapserouter(timestep)
host.kernels_per_fpga[1][4].run_neuroring(timestep)
host.kernels_per_fpga[1][4].run_synapserouter(timestep)

host.kernels_per_fpga[0][0].wait_for_kernel()
host.kernels_per_fpga[0][1].wait_for_kernel()
host.kernels_per_fpga[0][2].wait_for_kernel()
host.kernels_per_fpga[0][3].wait_for_kernel()
host.kernels_per_fpga[0][4].wait_for_kernel()
host.kernels_per_fpga[1][0].wait_for_kernel()
host.kernels_per_fpga[1][1].wait_for_kernel()
host.kernels_per_fpga[1][2].wait_for_kernel()
host.kernels_per_fpga[1][3].wait_for_kernel()
host.kernels_per_fpga[1][4].wait_for_kernel()
end_time = time.time()

print(f"time execution: {end_time - start_time}")

time execution: 0.738234281539917
